# 06 — Validación manual de la clasificación del estado de propiedades

## 1. Objetivo

Evaluar la confiabilidad de la clasificación automática del estado de los departamentos publicados en CABA mediante una revisión manual de una muestra de 50 propiedades.

Se busca identificar posibles errores, ambigüedades y limitaciones de las reglas textuales utilizadas para clasificar los inmuebles.

La validación contempla:

1. Selección reproducible de una muestra de publicaciones de diferentes categorías.
2. Clasificación independiente por parte de dos evaluadores, utilizando el título y la descripción de cada publicación.
3. Medición del acuerdo entre evaluadores mediante el coeficiente Cohen's kappa.
4. Construcción de una clasificación manual de referencia mediante la revisión y adjudicación de los desacuerdos por parte del evaluador A.
5. Comparación entre la clasificación manual de referencia y la clasificación automática.

Esta evaluación complementa los análisis de robustez de la notebook 05 y permite identificar oportunidades de mejora en las reglas de clasificación.

**Alcance:** los resultados describen el desempeño sobre la muestra seleccionada. Dado que se utilizó un muestreo estratificado no proporcional, las métricas no deben interpretarse directamente como estimaciones representativas del conjunto de propiedades analizadas.

In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

ruta = Path("data/processed/remax_deptos_features.csv")

assert ruta.exists(), f"No se encontró el archivo: {ruta}"

df = pd.read_csv(ruta)

print("Dimensiones:", df.shape)

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nDistribución de estados:")
print(df["Estado_Propiedad"].value_counts(dropna=False) if "Estado_Propiedad" in df.columns else "Verificar nombre de columna")



Dimensiones: (9499, 100)

Columnas disponibles:
['ID', 'ID_Entidad', 'Operacion', 'Tipo_Propiedad', 'Titulo', 'Descripcion', 'Fecha_Publicacion', 'Antiguedad', 'Apto_Credito', 'Amenities', 'Moneda', 'Precio_Valor', 'Moneda_Expensas', 'Expensas', 'Precio_m2_Total', 'Precio_m2_Cubierto', 'Ambientes', 'Dormitorios', 'Baños', 'Superficie_Total_m2', 'Superficie_Cubierta_m2', 'Direccion', 'Barrio', 'Ubicacion', 'Latitud', 'Longitud', 'Estado_Publicacion', 'ID_Interno', 'Link', 'Tipo_Propiedad_Original', 'Ambientes_Titulo', 'Flag_Ambientes_Dudoso', 'Flag_Dormitorios_Inconsistente', 'Flag_Posible_Republicacion', 'Expensas_Informadas', 'Antiguedad_Informada', 'Amenities_Informadas', 'Flag_Atipico_Bajo', 'Flag_Atipico_Alto', 'Flag_Extremo', 'Flag_Outlier_Multivariado', 'Barrio_Oficial', 'Comuna', 'Flag_Coordenadas_Invalidas', 'Luminoso', 'Vista', 'Silencioso', 'Contrafrente', 'Al_Frente', 'Apto_Profesional', 'Reciclado_Texto', 'A_Refaccionar', 'Pileta', 'SUM', 'Parrilla', 'Gimnasio', 'Laundry', 

## 2. Selección de la muestra para validación manual

Se selecciona una muestra aleatoria estratificada de 50 publicaciones, utilizando como estratos las cuatro categorías de `Estado_Propiedad`.

La distribución prioriza la representación de todas las categorías, especialmente aquellas con menor frecuencia en el dataset.

Se utiliza una semilla fija (`random_state=42`) para garantizar la reproducibilidad.

La muestra no es proporcional a la distribución del dataset completo. Por lo tanto, las métricas obtenidas permitirán evaluar el comportamiento de la clasificación en los distintos estratos, pero no estimar directamente la tasa global de error del conjunto de publicaciones.

In [38]:

# Cantidad de publicaciones por categoría
n_por_estado = {
    "Sin_clasificar": 15,
    "Buen_estado": 12,
    "Reciclada_refaccionada": 12,
    "Necesita_obra": 11
}

# Muestreo aleatorio estratificado reproducible
muestras = []

for estado, n in n_por_estado.items():
    grupo = df[df["Estado_Propiedad"] == estado]

    muestra = grupo.sample(
        n=n,
        random_state=42
    )

    muestras.append(muestra)

df_muestra = pd.concat(
    muestras,
    ignore_index=True
)

# Mezclar el orden para evitar agrupar por estado
df_muestra = df_muestra.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Identificador de revisión
df_muestra.insert(
    0,
    "ID_Revision",
    range(1, len(df_muestra) + 1)
)

print("Total de propiedades seleccionadas:", len(df_muestra))

print("\nDistribución por estado automático:")
print(df_muestra["Estado_Propiedad"].value_counts())

print("\nIDs únicos:", df_muestra["ID"].nunique())

display(
    df_muestra[
        ["ID_Revision", "ID", "Barrio", "Estado_Propiedad", "Titulo"]
    ].head(10)
)


Total de propiedades seleccionadas: 50

Distribución por estado automático:
Estado_Propiedad
Sin_clasificar            15
Reciclada_refaccionada    12
Buen_estado               12
Necesita_obra             11
Name: count, dtype: int64

IDs únicos: 50


,ID_Revision,ID,Barrio,Estado_Propiedad,Titulo
0,1,631965,La Paternal,Sin_clasificar,VENTA | Departamento 3 Ambientes en La Paternal
1,2,596310,Flores,Necesita_obra,3 ambientes a refaccionar (oportunidad)
2,3,571049,Balvanera,Reciclada_refaccionada,VENTA DEPTO 2 AMB- BALVANERA
3,4,621867,San Nicolás,Necesita_obra,OFICINA 113m² DIVISIBLE VENTA FLIPPING AirBnb
4,5,621167,San Nicolás,Buen_estado,Departamento 2 ambientes en San Nicolas
5,6,511218,Belgrano,Necesita_obra,Depto 152m² 4Amb Dep Belgrano Cabildo Oportunidad
6,7,617104,Barrio Norte,Buen_estado,VENTA- 3 AMBIENTES - BARRIO NORTE - APTO CREDITO
7,8,576407,Parque Chacabuco,Buen_estado,VENTA MONOAMBIENTE EN CABALLITO CON BALCÓN
8,9,612989,San Nicolás,Reciclada_refaccionada,VENTA MONOAMBIENTE - APTO PROFESIONAL
9,10,530816,Villa Pueyrredón,Buen_estado,DEPARTAMENTO 2 AMBIENTES VENTA VILLA PUEYRREDÓN


## 3. Preparación de la revisión manual

Se prepararon dos archivos de evaluación con las mismas 50 publicaciones, conservando el identificador, título, descripción y enlace original.

Las plantillas no incluían la clasificación automática, con el objetivo de evitar que esta condicionara las decisiones manuales.

Se utilizaron cuatro categorías:

- **Buen_estado:** la publicación indica explícitamente que el inmueble se encuentra en buenas condiciones.
- **Reciclada_refaccionada:** existen evidencias de una renovación o refacción ya realizada.
- **Necesita_obra:** se menciona explícitamente la necesidad de refacción, renovación o reparación.
- **Sin_clasificar:** la información es insuficiente, ambigua o contradictoria para determinar el estado.

Se conservaron las dos series de etiquetas manuales para medir su concordancia y posteriormente revisar los casos en los que existían desacuerdos.

Los archivos completos se encuentran en `data/validation/`.

In [39]:

from pathlib import Path

# Columnas visibles para los evaluadores
columnas_revision = [
    "ID_Revision",
    "ID",
    "Titulo",
    "Descripcion",
    "Link"
]

# Crear plantilla sin revelar la clasificación automática
plantilla = df_muestra[columnas_revision].copy()

plantilla["Estado_Manual"] = ""
plantilla["Observaciones"] = ""

# Carpeta de salida
carpeta_salida = Path("data/validation")
carpeta_salida.mkdir(parents=True, exist_ok=True)

# Archivos para revisión independiente
for evaluador in ["A", "B"]:
    ruta_salida = carpeta_salida / f"validacion_evaluador_{evaluador}.csv"

    plantilla.to_csv(
        ruta_salida,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Archivo generado: {ruta_salida}")

print("\nVista previa:")
display(plantilla[["ID_Revision", "ID", "Titulo", "Estado_Manual"]].head())


Archivo generado: data/validation/validacion_evaluador_A.csv
Archivo generado: data/validation/validacion_evaluador_B.csv

Vista previa:


,ID_Revision,ID,Titulo,Estado_Manual
0,1,631965,VENTA | Departamento 3 Ambientes en La Paternal,
1,2,596310,3 ambientes a refaccionar (oportunidad),
2,3,571049,VENTA DEPTO 2 AMB- BALVANERA,
3,4,621867,OFICINA 113m² DIVISIBLE VENTA FLIPPING AirBnb,
4,5,621167,Departamento 2 ambientes en San Nicolas,


## 4. Trazabilidad y exportación de la muestra

Se conserva una copia de la muestra seleccionada con las etiquetas automáticas originales para permitir su posterior comparación con las clasificaciones manuales.

Los archivos destinados a los evaluadores no incluyen dichas etiquetas, evitando influir en sus decisiones.

La muestra se mantiene fija durante todo el proceso de validación, utilizando los identificadores `ID` e `ID_Revision` para vincular las evaluaciones y calcular las métricas de concordancia.

In [40]:

from google.colab import files

# Guardar muestra original con clasificación automática
ruta_muestra = carpeta_salida / "muestra_validacion_50.csv"

df_muestra.to_csv(
    ruta_muestra,
    index=False,
    encoding="utf-8-sig"
)

print("Muestra original guardada:", ruta_muestra)

# Descargar archivos para los evaluadores
#files.download(str(carpeta_salida / "validacion_evaluador_A.csv"))
#files.download(str(carpeta_salida / "validacion_evaluador_B.csv"))

# Descargar también la muestra original para conservarla
#files.download(str(ruta_muestra))


Muestra original guardada: data/validation/muestra_validacion_50.csv


## 5. Concordancia entre evaluadores

Se comparan las clasificaciones manuales realizadas de manera independiente por dos evaluadores distintos sobre las mismas 50 publicaciones.

Se calculan dos indicadores:

- **Porcentaje de acuerdo observado:** proporción de propiedades para las cuales ambos evaluadores asignaron la misma categoría.
- **Cohen's kappa:** mide el acuerdo entre evaluadores, ajustando por la concordancia esperada por azar según las frecuencias de las categorías.

También se construye una matriz de concordancia para identificar las categorías que generan mayores diferencias de interpretación.

Los desacuerdos se analizaron posteriormente mediante una revisión adicional realizada por el evaluador A, quien adjudicó la clasificación manual definitiva de cada caso, sin modificar las respuestas originales de ninguno de los evaluadores.

In [41]:

from sklearn.metrics import cohen_kappa_score, confusion_matrix

carpeta = Path("data/validation")

a = pd.read_csv(carpeta / "validacion_evaluador_A_clasificado.csv")
b = pd.read_csv(carpeta / "validacion_evaluador_B_clasificado.csv")

comparacion = a[
    ["ID_Revision", "ID", "Estado_Manual"]
].merge(
    b[["ID_Revision", "ID", "Estado_Manual"]],
    on=["ID_Revision", "ID"],
    suffixes=("_A", "_B"),
    validate="one_to_one"
)

assert len(comparacion) == 50, "La comparación no contiene 50 propiedades"
assert comparacion[["Estado_Manual_A", "Estado_Manual_B"]].notna().all().all()

acuerdo = (
    comparacion["Estado_Manual_A"]
    == comparacion["Estado_Manual_B"]
).mean()

kappa = cohen_kappa_score(
    comparacion["Estado_Manual_A"],
    comparacion["Estado_Manual_B"]
)

print(f"Propiedades comparadas: {len(comparacion)}")
print(f"Acuerdo observado: {acuerdo:.2%}")
print(f"Cohen's kappa: {kappa:.4f}")

print("\nMatriz de concordancia:")
display(pd.crosstab(
    comparacion["Estado_Manual_A"],
    comparacion["Estado_Manual_B"],
    rownames=["Evaluador A"],
    colnames=["Evaluador B"]
))


Propiedades comparadas: 50
Acuerdo observado: 66.00%
Cohen's kappa: 0.5398

Matriz de concordancia:


Evaluador B,Buen_estado,Necesita_obra,Reciclada_refaccionada,Sin_clasificar
Evaluador A,,,,
Buen_estado,15,0,1,0
Necesita_obra,0,9,0,0
Reciclada_refaccionada,3,0,6,0
Sin_clasificar,8,2,3,3


## 6. Análisis de desacuerdos entre evaluadores

La comparación entre evaluadores presenta un acuerdo observado del 66% (33 de 50 propiedades) y un coeficiente Cohen's kappa de 0,5398, correspondiente a una concordancia moderada según la escala convencional de Landis y Koch.

Se identificaron 17 propiedades con clasificaciones diferentes.

La mayor fuente de discrepancias corresponde a publicaciones que un evaluador considera insuficientes para determinar el estado, mientras que el otro identifica evidencia para asignar una categoría específica.

Estos 17 casos fueron sometidos a una revisión posterior por parte del evaluador A, utilizando los títulos y las descripciones originales, con el objetivo de adjudicar una etiqueta manual definitiva.

La adjudicación se realizó mediante revisión y justificación de cada desacuerdo, sin utilizar la clasificación automática como criterio de decisión. Las limitaciones de este procedimiento se documentan en la sección 7.

In [42]:

# Identificar propiedades con desacuerdo
desacuerdos = comparacion[
    comparacion["Estado_Manual_A"]
    != comparacion["Estado_Manual_B"]
].copy()

# Agregar información textual para la revisión
desacuerdos = desacuerdos.merge(
    df_muestra[
        ["ID_Revision", "ID", "Titulo", "Descripcion", "Link"]
    ],
    on=["ID_Revision", "ID"],
    how="left",
    validate="one_to_one"
)

# Columnas para completar después de la revisión
desacuerdos["Estado_Consenso"] = ""
desacuerdos["Justificacion_Consenso"] = ""

desacuerdos = desacuerdos.sort_values("ID_Revision")

print("Total de desacuerdos:", len(desacuerdos))

display(
    desacuerdos[
        [
            "ID_Revision",
            "ID",
            "Titulo",
            "Estado_Manual_A",
            "Estado_Manual_B"
        ]
    ]
)


Total de desacuerdos: 17


,ID_Revision,ID,Titulo,Estado_Manual_A,Estado_Manual_B
0,4,621867,OFICINA 113m² DIVISIBLE VENTA FLIPPING AirBnb,Sin_clasificar,Necesita_obra
1,5,621167,Departamento 2 ambientes en San Nicolas,Buen_estado,Reciclada_refaccionada
2,9,612989,VENTA MONOAMBIENTE - APTO PROFESIONAL,Sin_clasificar,Buen_estado
3,12,600833,VENTA 2 AMBIENTES BALCON CORRIDO Y PILETA,Sin_clasificar,Buen_estado
4,14,537509,Venta departamento 4 ambientes/pileta propia.,Sin_clasificar,Reciclada_refaccionada
5,15,636437,Departamento Venta 2 Ambientes Floresta 2 Patios,Sin_clasificar,Reciclada_refaccionada
6,20,470032,DEPTO VENTA 2 AMB BELGRANO BALCON VISTA ABIERTA,Sin_clasificar,Buen_estado
7,21,640970,DEPARTAMENTO VENTA RECOLETA 4 AMB COCHERA SEG ...,Sin_clasificar,Buen_estado
8,25,653917,Venta departamento 3 Ambientes - VILLA URQUIZA,Sin_clasificar,Reciclada_refaccionada
9,26,535835,Venta 4 amb palermo parrilla y terraza 2 coch,Sin_clasificar,Buen_estado


## 7. Resolución manual de desacuerdos

Se identificaron 17 publicaciones con clasificaciones diferentes entre las evaluaciones A y B, equivalentes al 34% de la muestra.

Para construir una clasificación manual de referencia, se revisaron nuevamente estas publicaciones considerando las evidencias presentes en el título y la descripción.

La revisión se realizó siguiendo los siguientes criterios:

- Utilizar las mismas cuatro categorías definidas para la validación.
- Fundamentar las decisiones mediante evidencia textual.
- Asignar `Sin_clasificar` cuando la información fuera insuficiente o contradictoria.
- Evitar utilizar la clasificación automática como referencia para resolver los desacuerdos.

De los 17 casos revisados, la clasificación definitiva fue:

| Categoría | Cantidad |
|---|---:|
| Sin_clasificar | 13 |
| Reciclada_refaccionada | 3 |
| Buen_estado | 1 |
| Necesita_obra | 0 |
| **Total** | **17** |

En las otras 33 publicaciones se conservó la categoría compartida por ambas evaluaciones.

Las decisiones definitivas y sus justificaciones se documentaron en `data/validation/revision_consenso_17_completo.csv`.

Esta revisión permitió construir una referencia manual completa para las 50 publicaciones, utilizada posteriormente para evaluar la clasificación automática.

La resolución de los 17 desacuerdos estuvo a cargo del evaluador A, quien realizó una segunda revisión de las publicaciones considerando las justificaciones textuales. En los 17 casos, la decisión definitiva coincidió con su clasificación inicial. Por lo tanto, la referencia manual debe interpretarse como una clasificación adjudicada por el evaluador A, luego de contrastarla con una segunda evaluación independiente, y no como el resultado de un consenso entre ambos evaluadores. Esta característica constituye una limitación metodológica de la validación.

In [43]:

# Preparar archivo de revisión de los 17 desacuerdos
columnas_consenso = [
    "ID_Revision",
    "ID",
    "Titulo",
    "Descripcion",
    "Link",
    "Estado_Manual_A",
    "Estado_Manual_B",
    "Estado_Consenso",
    "Justificacion_Consenso"
]

revision_consenso = desacuerdos[columnas_consenso].copy()

ruta_consenso = carpeta_salida / "revision_consenso_17.csv"

revision_consenso.to_csv(
    ruta_consenso,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:", ruta_consenso)
print("Propiedades pendientes de consenso:", len(revision_consenso))

from google.colab import files
#files.download(str(ruta_consenso))


Archivo generado: data/validation/revision_consenso_17.csv
Propiedades pendientes de consenso: 17


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Construcción de la clasificación manual de referencia

Se construye una clasificación manual de referencia para las 50 publicaciones seleccionadas.

Para las 33 propiedades en las que ambos evaluadores coincidieron, se conserva la categoría compartida. Para las 17 propiedades con desacuerdos, se utiliza la categoría definida durante la revisión posterior, acompañada de su justificación.

Esta clasificación se utilizará como referencia para evaluar las reglas automáticas de detección del estado de las propiedades.

La revisión se realiza sin utilizar las etiquetas automáticas como criterio de decisión. Los resultados permiten evaluar el desempeño sobre la muestra seleccionada, pero no representan directamente la precisión global del modelo sobre las 9.499 publicaciones.

In [44]:

# Cargar las 17 decisiones revisadas
ruta_consenso = Path(
    "data/validation/revision_consenso_17_completo.csv"
)

consenso = pd.read_csv(ruta_consenso)

# Validaciones
assert len(consenso) == 17
assert consenso["ID"].is_unique
assert consenso["Estado_Consenso"].notna().all()

# Unir las decisiones con las 50 evaluaciones
validacion = comparacion.merge(
    consenso[["ID", "Estado_Consenso"]],
    on="ID",
    how="left",
    validate="one_to_one"
)

# Conservar los acuerdos y reemplazar los desacuerdos
coinciden = (
    validacion["Estado_Manual_A"]
    == validacion["Estado_Manual_B"]
)

validacion["Estado_Referencia"] = np.where(
    coinciden,
    validacion["Estado_Manual_A"],
    validacion["Estado_Consenso"]
)

assert validacion["Estado_Referencia"].notna().all()
assert len(validacion) == 50

# Agregar clasificación automática original
validacion = validacion.merge(
    df_muestra[["ID", "Estado_Propiedad"]],
    on="ID",
    how="left",
    validate="one_to_one"
)

print("Propiedades validadas:", len(validacion))

print("\nDistribución manual definitiva:")
print(validacion["Estado_Referencia"].value_counts())

print("\nDistribución automática:")
print(validacion["Estado_Propiedad"].value_counts())


Propiedades validadas: 50

Distribución manual definitiva:
Estado_Referencia
Buen_estado               16
Sin_clasificar            16
Necesita_obra              9
Reciclada_refaccionada     9
Name: count, dtype: int64

Distribución automática:
Estado_Propiedad
Sin_clasificar            15
Reciclada_refaccionada    12
Buen_estado               12
Necesita_obra             11
Name: count, dtype: int64


## 9. Evaluación de la clasificación automática

Se compara la clasificación automática de las 50 propiedades con la clasificación manual de referencia obtenida en la etapa anterior.

Se calculan las siguientes métricas:

- **Accuracy:** proporción de publicaciones clasificadas correctamente.
- **Precision:** proporción de predicciones de una categoría que coinciden con la referencia manual.
- **Recall:** proporción de publicaciones de cada categoría manual que el modelo identifica correctamente.
- **F1-score:** media armónica entre precision y recall.
- **Cohen's kappa:** concordancia entre las etiquetas automáticas y la referencia manual, ajustada por el acuerdo esperado por azar.

También se construye una matriz de confusión para identificar los errores más frecuentes.

Dado que se utilizó una muestra estratificada no proporcional, estas métricas describen el desempeño sobre las 50 publicaciones seleccionadas y no deben interpretarse directamente como estimaciones del rendimiento global sobre las 9.499 propiedades.

La clasificación manual se utiliza como referencia operativa, reconociendo que también puede presentar errores o ambigüedades.

In [45]:

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    cohen_kappa_score,
    confusion_matrix
)

categorias = [
    "Buen_estado",
    "Reciclada_refaccionada",
    "Necesita_obra",
    "Sin_clasificar"
]

y_real = validacion["Estado_Referencia"]
y_pred = validacion["Estado_Propiedad"]

# Métricas generales
accuracy = accuracy_score(y_real, y_pred)
kappa_modelo = cohen_kappa_score(y_real, y_pred)

# Métricas por categoría
precision, recall, f1, soporte = precision_recall_fscore_support(
    y_real,
    y_pred,
    labels=categorias,
    zero_division=0
)

metricas = pd.DataFrame({
    "Categoria": categorias,
    "Precision": precision,
    "Recall": recall,
    "F1_score": f1,
    "Soporte": soporte
})

# Matriz de confusión
matriz = pd.DataFrame(
    confusion_matrix(y_real, y_pred, labels=categorias),
    index=categorias,
    columns=categorias
)

print(f"Accuracy: {accuracy:.2%}")
print(f"Cohen's kappa (manual vs automático): {kappa_modelo:.4f}")

print("\nMétricas por categoría:")
display(metricas.round(3))

print("\nMatriz de confusión:")
print("Filas: referencia manual | Columnas: modelo automático")
display(matriz)


Accuracy: 78.00%
Cohen's kappa (manual vs automático): 0.7045

Métricas por categoría:


,Categoria,Precision,Recall,F1_score,Soporte
0,Buen_estado,0.917,0.688,0.786,16
1,Reciclada_refaccionada,0.750,1.000,0.857,9
2,Necesita_obra,0.818,1.000,0.900,9
3,Sin_clasificar,0.667,0.625,0.645,16



Matriz de confusión:
Filas: referencia manual | Columnas: modelo automático


,Buen_estado,Reciclada_refaccionada,Necesita_obra,Sin_clasificar
Buen_estado,11,0,0,5
Reciclada_refaccionada,0,9,0,0
Necesita_obra,0,0,9,0
Sin_clasificar,1,3,2,10


## 10. Análisis de errores de clasificación

La comparación con la referencia manual muestra un accuracy del 78% (39 de 50 propiedades) y un Cohen's kappa de 0,7045.

El modelo identificó correctamente las 9 propiedades de la muestra cuya referencia manual indica `Necesita_obra`, aunque asignó esta categoría a otras 2 publicaciones consideradas insuficientemente documentadas.

Asimismo, identificó correctamente las 9 propiedades manualmente clasificadas como `Reciclada_refaccionada`, pero asignó esa etiqueta a 3 publicaciones que la revisión manual dejó sin clasificar.

La principal dificultad se concentra en la distinción entre publicaciones con evidencia suficiente para asignar un estado y publicaciones con información ambigua o incompleta.

Se analizarán individualmente las 11 discrepancias para identificar patrones de error y proponer mejoras en las reglas de clasificación, sin modificar retroactivamente las etiquetas utilizadas para esta validación.

Los resultados corresponden a una muestra estratificada no proporcional, por lo que no representan directamente el desempeño global del clasificador.

In [46]:

# Identificar discrepancias entre referencia manual y modelo
errores_modelo = validacion[
    validacion["Estado_Referencia"]
    != validacion["Estado_Propiedad"]
].copy()

# Agregar título, descripción y enlace
errores_modelo = errores_modelo.merge(
    df_muestra[
        ["ID", "Titulo", "Descripcion", "Link"]
    ],
    on="ID",
    how="left",
    validate="one_to_one"
)

errores_modelo = errores_modelo.sort_values("ID_Revision")

print("Total de errores del modelo:", len(errores_modelo))

print("\nTipos de discrepancia:")
display(
    errores_modelo.groupby(
        ["Estado_Referencia", "Estado_Propiedad"]
    ).size().reset_index(name="Cantidad")
)

print("\nPropiedades con discrepancias:")
display(
    errores_modelo[
        [
            "ID_Revision",
            "ID",
            "Titulo",
            "Estado_Referencia",
            "Estado_Propiedad"
        ]
    ]
)


Total de errores del modelo: 11

Tipos de discrepancia:


,Estado_Referencia,Estado_Propiedad,Cantidad
0,Buen_estado,Sin_clasificar,5
1,Sin_clasificar,Buen_estado,1
2,Sin_clasificar,Necesita_obra,2
3,Sin_clasificar,Reciclada_refaccionada,3



Propiedades con discrepancias:


,ID_Revision,ID,Titulo,Estado_Referencia,Estado_Propiedad
0,1,631965,VENTA | Departamento 3 Ambientes en La Paternal,Buen_estado,Sin_clasificar
1,4,621867,OFICINA 113m² DIVISIBLE VENTA FLIPPING AirBnb,Sin_clasificar,Necesita_obra
2,9,612989,VENTA MONOAMBIENTE - APTO PROFESIONAL,Sin_clasificar,Reciclada_refaccionada
3,16,545832,Venta depto 4 amb en San Cristobal full amenities,Buen_estado,Sin_clasificar
4,20,470032,DEPTO VENTA 2 AMB BELGRANO BALCON VISTA ABIERTA,Sin_clasificar,Buen_estado
5,25,653917,Venta departamento 3 Ambientes - VILLA URQUIZA,Sin_clasificar,Reciclada_refaccionada
6,33,646520,Venta departamento Av. Libertador ¨GRAN VISTA ¨,Sin_clasificar,Reciclada_refaccionada
7,36,547397,Venta departamento 1 amb Balcón Nuñez APTO CRE...,Buen_estado,Sin_clasificar
8,41,629155,VENTA MONOAMBIENTE CON PATIO COGHLAN,Buen_estado,Sin_clasificar
9,46,624091,Venta Monoambiente Abasto con amenities,Buen_estado,Sin_clasificar


## 11. Implicancias de los errores para la identificación de oportunidades

La validación manual identificó 11 discrepancias entre las etiquetas automáticas y la referencia manual.

Dos publicaciones fueron clasificadas automáticamente como `Necesita_obra`, mientras que la revisión manual consideró que no existía evidencia suficiente para determinar su estado.

Este tipo de discrepancia es especialmente relevante para el objetivo del proyecto, dado que la clasificación del estado interviene en la selección de propiedades candidatas a refacción y en el cálculo del Precio Esperado Reciclado (PER).

Se analizará si estas publicaciones recibieron estimaciones de PER y qué potencial bruto de inversión presentan.

El objetivo no es extrapolar el número de errores al universo de propiedades, sino identificar riesgos concretos que permitan mejorar los controles del proceso de selección de oportunidades.

In [47]:

# Falsos positivos de Necesita_obra respecto de la referencia manual
falsos_positivos_obra = errores_modelo[
    (errores_modelo["Estado_Propiedad"] == "Necesita_obra")
    & (errores_modelo["Estado_Referencia"] != "Necesita_obra")
].copy()

# Recuperar métricas económicas originales
columnas_economicas = [
    "ID",
    "Barrio",
    "Precio_Valor",
    "PER_Base",
    "PER_N_Comparables",
    "PER_Nivel",
    "Potencial_Bruto_Base_pct"
]

impacto = falsos_positivos_obra[
    ["ID", "Titulo", "Estado_Referencia", "Estado_Propiedad"]
].merge(
    df_muestra[columnas_economicas],
    on="ID",
    how="left",
    validate="one_to_one"
)

print("Falsos positivos de Necesita_obra:", len(impacto))

print(
    "Con PER calculado:",
    impacto["PER_Base"].notna().sum()
)

display(impacto)


Falsos positivos de Necesita_obra: 2
Con PER calculado: 2


,ID,Titulo,Estado_Referencia,Estado_Propiedad,Barrio,Precio_Valor,PER_Base,PER_N_Comparables,PER_Nivel,Potencial_Bruto_Base_pct
0,621867,OFICINA 113m² DIVISIBLE VENTA FLIPPING AirBnb,Sin_clasificar,Necesita_obra,San Nicolás,135000.0,170687.022901,15.0,3.0,26.434832
1,599923,Depto 3 amb. con patio en pb – San Cristobal,Sin_clasificar,Necesita_obra,San Cristobal,65000.0,71438.888889,37.0,3.0,9.905983


## 12. Conclusiones y recomendaciones

### 12.1. Resultados de la validación manual

Se evaluó la clasificación automática del estado de 50 publicaciones seleccionadas mediante un muestreo aleatorio estratificado no proporcional.

La comparación entre las dos evaluaciones manuales arrojó un acuerdo observado del **66%** y un coeficiente **Cohen's kappa de 0,5398**, correspondiente a una concordancia moderada según la escala convencional de Landis y Koch.

Luego de revisar los 17 desacuerdos, se construyó una clasificación manual de referencia para las 50 publicaciones.

Al comparar esta referencia con el modelo automático, se obtuvieron los siguientes resultados:

- **Accuracy:** 78% (39 de 50 clasificaciones coincidentes).
- **Cohen's kappa:** 0,7045.
- **Recall de `Necesita_obra`:** 100% (9 de 9).
- **Precision de `Necesita_obra`:** 81,8% (9 de 11).
- **Recall de `Reciclada_refaccionada`:** 100% (9 de 9).
- **Recall de `Sin_clasificar`:** 62,5% (10 de 16).

Estos resultados indican un desempeño favorable sobre la muestra evaluada, aunque evidencian dificultades para distinguir publicaciones con información suficiente de aquellas con descripciones ambiguas.

### 12.2. Principales errores detectados

Se identificaron 11 discrepancias entre la clasificación automática y la referencia manual:

- Cinco propiedades manualmente consideradas en buen estado quedaron sin clasificar.
- Tres publicaciones sin evidencia manual suficiente fueron clasificadas como recicladas o refaccionadas.
- Dos publicaciones sin evidencia manual suficiente fueron clasificadas como necesitadas de obra.
- Una publicación sin evidencia manual suficiente fue clasificada en buen estado.

Los errores sugieren la necesidad de revisar el contexto de las expresiones textuales, particularmente cuando las mejoras mencionadas corresponden al edificio, cuando existen referencias a renovaciones parciales o cuando la descripción no permite establecer claramente el estado de la unidad.

### 12.3. Impacto sobre la identificación de oportunidades de flipping

Los dos falsos positivos de `Necesita_obra` recibieron estimaciones de Precio Esperado Reciclado (PER):

| ID | Barrio | Precio publicado (USD) | PER base (USD) | Potencial bruto |
|---|---|---:|---:|---:|
| 621867 | San Nicolás | 135.000 | 170.687 | 26,43% |
| 599923 | San Cristóbal | 65.000 | 71.439 | 9,91% |

El caso 621867 supera el umbral preliminar del 25% de potencial bruto. No obstante, su estado no pudo confirmarse mediante la revisión manual y el título de la publicación menciona una oficina.

Esto demuestra que una clasificación automática incorrecta puede afectar la selección de candidatos y generar oportunidades aparentes que requieren verificación adicional.

### 12.4. Recomendaciones para el proyecto

1. **Incorporar una revisión manual obligatoria** antes de recomendar oportunidades de inversión priorizadas.
2. **Distinguir el estado de la unidad del estado del edificio**, evitando interpretar mejoras de espacios comunes como refacciones del departamento.
3. **Revisar las reglas de clasificación textual** para reducir asignaciones basadas en expresiones ambiguas o fuera de contexto.
4. **Agregar controles de elegibilidad del inmueble**, especialmente para publicaciones que mencionen oficinas u otros usos diferentes al residencial.
5. **Conservar indicadores de incertidumbre y trazabilidad**, permitiendo identificar qué publicaciones requieren verificación adicional.
6. **Evaluar la rentabilidad neta**, incorporando costos de compra, refacción, venta y riesgos, en lugar de utilizar únicamente el potencial bruto.

### 12.5. Limitaciones

La muestra de 50 publicaciones fue seleccionada mediante estratificación no proporcional, por lo que los resultados no deben extrapolarse directamente al conjunto de 9.499 propiedades.

La clasificación manual también está sujeta a ambigüedades e interpretaciones, especialmente cuando las publicaciones presentan información incompleta.

Las métricas obtenidas describen la concordancia con la referencia manual construida, no una verificación física del estado de los inmuebles.

En consecuencia, esta validación fortalece la interpretación de los resultados del proyecto, pero no reemplaza una inspección profesional previa a una decisión de inversión.

### 12.6. Conclusión general

La validación manual permitió cuantificar el desempeño del clasificador, identificar errores concretos y comprobar que algunos de ellos alcanzan la etapa de estimación económica del PER.

El modelo constituye una herramienta útil para filtrar y analizar publicaciones inmobiliarias, pero sus resultados deben interpretarse como señales preliminares y no como recomendaciones automáticas de compra.

La siguiente etapa del proyecto deberá transformar estas señales en un ranking de oportunidades de flipping que considere rentabilidad neta, costos e incertidumbre.

## 13. Exportación de resultados y reproducibilidad

Se exporta la clasificación manual de referencia de las 50 publicaciones, conservando los identificadores originales, las etiquetas de ambos evaluadores, la clasificación definitiva y la etiqueta automática.

El archivo permite reproducir las métricas de validación y analizar las discrepancias sin necesidad de repetir la revisión manual.

Se verifica que las 50 propiedades tengan identificadores únicos, categorías válidas y clasificaciones completas.

El resultado se guarda en `data/validation/validacion_final_50.csv` para su incorporación al repositorio.

In [48]:
from pathlib import Path

categorias_validas = {
    "Buen_estado",
    "Reciclada_refaccionada",
    "Necesita_obra",
    "Sin_clasificar"
}

# Seleccionar las columnas finales
columnas_finales = [
    "ID_Revision",
    "ID",
    "Estado_Manual_A",
    "Estado_Manual_B",
    "Estado_Referencia",
    "Estado_Propiedad"
]

resultado_final = validacion[columnas_finales].copy()

# Validaciones de integridad
assert len(resultado_final) == 50
assert resultado_final["ID"].is_unique
assert resultado_final["ID_Revision"].is_unique
assert resultado_final.notna().all().all()

for columna in [
    "Estado_Manual_A",
    "Estado_Manual_B",
    "Estado_Referencia",
    "Estado_Propiedad"
]:
    assert set(resultado_final[columna]).issubset(categorias_validas)

# Guardar resultado
ruta_final = Path("data/validation/validacion_final_50.csv")
ruta_final.parent.mkdir(parents=True, exist_ok=True)

resultado_final.to_csv(
    ruta_final,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:", ruta_final)
print("Total de propiedades:", len(resultado_final))
print("IDs únicos:", resultado_final["ID"].nunique())

print("\nDistribución final:")
print(resultado_final["Estado_Referencia"].value_counts())

print("\nVista previa:")
display(resultado_final.head())

# Descargar a la computadora
from google.colab import files
#files.download(str(ruta_final))


Archivo generado: data/validation/validacion_final_50.csv
Total de propiedades: 50
IDs únicos: 50

Distribución final:
Estado_Referencia
Buen_estado               16
Sin_clasificar            16
Necesita_obra              9
Reciclada_refaccionada     9
Name: count, dtype: int64

Vista previa:


,ID_Revision,ID,Estado_Manual_A,Estado_Manual_B,Estado_Referencia,Estado_Propiedad
0,1,631965,Buen_estado,Buen_estado,Buen_estado,Sin_clasificar
1,2,596310,Necesita_obra,Necesita_obra,Necesita_obra,Necesita_obra
2,3,571049,Reciclada_refaccionada,Reciclada_refaccionada,Reciclada_refaccionada,Reciclada_refaccionada
3,4,621867,Sin_clasificar,Necesita_obra,Sin_clasificar,Necesita_obra
4,5,621167,Buen_estado,Reciclada_refaccionada,Buen_estado,Buen_estado


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>